# Libs

In [ ]:
import pandas as pd
import numpy as np
import os

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"

# Load Data

In [ ]:
process_name = "CaO Liv"
timestamp = "Timestamp"

df_dataset = pd.read_csv(DIR_DATA + process_name + ".csv", sep=";", decimal=".", encoding="utf-8-sig")
df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")

list_columns_drop = ["AR.F1.SI12501", "AR.F1.CONS_CALOR", "AR.F1.PI52607",   "AR.F1.SAIDA_SILO_FARINHA_FSC_LIMS", "AR.F1.MFARINHA_FSC_LIMS"] # TAG AR.F1.SI12501 está retornando valores NaN e TAGs AR.F1.CONS_CALOR e AR.F1.PI52607 não possuem histórico e nem estão mapeadas em nenhum processo
df_dataset.drop(list_columns_drop, axis=1, inplace=True, errors='ignore')
df_dataset = df_dataset.sort_values(by=timestamp)
df_dataset.set_index(timestamp, inplace=True)
df_dataset

In [ ]:
target_col = 'AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS'

# Identifica as variáveis preditoras (remove o target da lista)
features = [col for col in df_dataset.columns if col != target_col]

print(f"Total de features: {len(features)}")
print(f"Total de registros: {df_dataset.shape[0]}")

In [ ]:
df_sistema = pd.read_csv(DIR_DATA + process_name + "_subsistema.csv", sep=";")
df_sistema

# Plot Variables

In [ ]:
def plot_subplots(dados, vars_plot1, vars_plot2):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)

    for var in vars_plot1:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines'), row=1, col=1)

    for var in vars_plot2:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines', line_color='black'), row=2, col=1)

    fig.update_layout(
        height=700, 
        hovermode="x unified",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15, 
            xanchor="center",
            x=0.5,
            title=None
        )
    )
    
    return fig

fig_subplot = plot_subplots(
    dados=df_dataset,  
    vars_plot1=['AR.F1.II12501'], 
    vars_plot2=['AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS']
)
fig_subplot.show()

# Pre-process

# Feature Engineering

In [ ]:
# Define as janelas de tempo em minutos
windows = [15, 30, 60, 180]

# DataFrame vazio para armazenar as novas features com o mesmo índice
df_features = pd.DataFrame(index=df_dataset.index)

for w in windows:
    # Calcula a média móvel (min_periods garante cálculo mesmo com falhas curtas de sensor)
    mean_cols = df_dataset[features].rolling(window=w, min_periods=w//2).mean()
    mean_cols.columns = [f'{col}_mean_{w}m' for col in features]
    
    # Calcula o desvio padrão móvel (captura a variância/oscilação do processo)
    std_cols = df_dataset[features].rolling(window=w, min_periods=w//2).std()
    std_cols.columns = [f'{col}_std_{w}m' for col in features]
    
    # Junta as novas colunas
    df_features = pd.concat([df_features, mean_cols, std_cols], axis=1)

# Adiciona a variável Target de volta ao dataframe de features
df_features[target_col] = df_dataset[target_col]

print("Featurea dataframe:", df_features.shape)

In [ ]:
# Identifica os platôs comparando o valor atual com o minuto anterior
# Se for igual ao anterior, transforma em NaN (reverte o forward-fill)
df_features.loc[df_features[target_col] == df_features[target_col].shift(1), target_col] = np.nan

# Remove as linhas que ficaram com NaN (os platôs e o período inicial das janelas móveis)
df_model = df_features.dropna(subset=[target_col]).copy()

print(f"Registros úteis para treinamento (apenas instantes de mudança real): {df_model.shape[0]}")

# Separa X e Y
X = df_model.drop(columns=[target_col])
y = df_model[target_col]

# Baseline

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Divisão temporal (80% treino, 20% teste)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

# Instancia e treina o modelo baseline
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predição no conjunto de teste
y_pred = rf_model.predict(X_test)

# Avaliação
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"--- Desempenho Baseline ---")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# (Opcional) Visualiza as features mais importantes encontradas pela árvore
importancias = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nTop 10 Features mais importantes:")
print(importancias.nlargest(10))

In [ ]:
# Cria a figura
fig = go.Figure()

# Adiciona a linha dos valores reais
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_test, 
    mode='lines+markers', 
    name='Real (Laboratório)',
    line=dict(color='blue')
))

# Adiciona a linha das predições
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_pred, 
    mode='lines+markers', 
    name='Predição (Baseline)',
    line=dict(color='red', dash='dot')
))

# Configurações de layout
fig.update_layout(
    title='Comparação: Variável Real vs Predição (Conjunto de Teste)',
    xaxis_title='Timestamp',
    yaxis_title='AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Exibe o plot
fig.show()

In [ ]:
import xgboost as xgb
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Seleção de Features (Mantendo apenas as mais importantes para reduzir ruído)
# Usamos uma árvore simples para identificar as variáveis que realmente importam
selector = SelectFromModel(RandomForestRegressor(n_estimators=50, random_state=42), max_features=15)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

selected_cols = X_train.columns[selector.get_support()]
print(f"Features retidas ({len(selected_cols)}): \n{selected_cols.tolist()}\n")

# 2. Treinamento com XGBoost (Lida melhor com picos e não-linearidades)
# Parâmetros conservadores para evitar overfitting nas 490 amostras
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=3,             # Árvores rasas
    learning_rate=0.05,      # Aprendizado lento
    subsample=0.8,           # Amostragem de linhas
    colsample_bytree=0.8,    # Amostragem de colunas
    random_state=42
)

xgb_model.fit(X_train_sel, y_train)

# 3. Predição e Avaliação
y_pred_xgb = xgb_model.predict(X_test_sel)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"--- Desempenho Melhorado (Feature Selection + XGBoost) ---")
print(f"MAE: {mae_xgb:.4f}")
print(f"RMSE: {rmse_xgb:.4f}")
print(f"R²: {r2_xgb:.4f}")

# Atualize a célula de plotagem (Plotly) trocando 'y_pred' por 'y_pred_xgb'

In [ ]:
# Cria a figura
fig = go.Figure()

# Adiciona a linha dos valores reais
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_test, 
    mode='lines+markers', 
    name='Real (Laboratório)',
    line=dict(color='blue')
))

# Adiciona a linha das predições
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_pred_xgb, 
    mode='lines+markers', 
    name='Predição (Baseline)',
    line=dict(color='red', dash='dot')
))

# Configurações de layout
fig.update_layout(
    title='Comparação: Variável Real vs Predição (Conjunto de Teste)',
    xaxis_title='Timestamp',
    yaxis_title='AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Exibe o plot
fig.show()